In [1]:
from z3 import *

ProductID = DeclareSort('ProductID')
RatingID  = DeclareSort('RatingID')
SellerID  = DeclareSort('SellerID')

Product = Datatype('Product')
Product.declare(
    'mk',
    ('pid', ProductID),
    ('name', StringSort()),
    ('price', RealSort()),
    ('quantity', IntSort()),
    ('sold', IntSort()),
    ('seller', SellerID)
)
Product = Product.create()

Rating = Datatype('Rating')
Rating.declare(
    'mk',
    ('rid', RatingID),
    ('pid', ProductID),
    ('score', IntSort()),
    ('comment', StringSort())
)
Rating = Rating.create()

Products = Function('Products', ProductID, Product)
ProductExists = Function('ProductExists', ProductID, BoolSort())

Ratings = Function('Ratings', RatingID, Rating)
RatingExists = Function('RatingExists', RatingID, BoolSort())


Products_next = Function('Products_next', ProductID, Product)
ProductExists_next = Function('ProductExists_next', ProductID, BoolSort())

Ratings_next = Function('Ratings_next', RatingID, Rating)
RatingExists_next = Function('RatingExists_next', RatingID, BoolSort())

pid = Const('pid', ProductID)
rid = Const('rid', RatingID)


def product_invariants(p):
    return And(
        Product.price(p) >= 0,
        Product.quantity(p) >= 0,
        Product.sold(p) >= 0,
        Length(Product.name(p)) > 0
    )

def rating_invariants(r):
    return And(
        Rating.score(r) >= 1,
        Rating.score(r) <= 5,
        Length(Rating.comment(r)) > 0
    )

def global_invariants():
    return And(
        ForAll(
            pid,
            Implies(
                ProductExists(pid),
                product_invariants(Products(pid))
            )
        ),
        ForAll(
            rid,
            Implies(
                RatingExists(rid),
                rating_invariants(Ratings(rid))
            )
        )
    )


pid_f = Const('pid_f', ProductID)
rid_f = Const('rid_f', RatingID)

def frame_products_unchanged(changed_pid):
    return ForAll(
        pid_f,
        Implies(
            pid_f != changed_pid,
            And(
                Products_next(pid_f) == Products(pid_f),
                ProductExists_next(pid_f) == ProductExists(pid_f)
            )
        )
    )

def frame_all_ratings_unchanged():
    return ForAll(
        rid_f,
        And(
            Ratings_next(rid_f) == Ratings(rid_f),
            RatingExists_next(rid_f) == RatingExists(rid_f)
        )
    )


pq = Int('pq')

purchase = And(
    ProductExists(pid),
    pq >= 0,
    pq <= Product.quantity(Products(pid)),

    ProductExists_next(pid),
    Products_next(pid) ==
        Product.mk(
            Product.pid(Products(pid)),
            Product.name(Products(pid)),
            Product.price(Products(pid)),
            Product.quantity(Products(pid)) - pq,
            Product.sold(Products(pid)) + pq,
            Product.seller(Products(pid))
        ),

    frame_products_unchanged(pid),
    frame_all_ratings_unchanged()
)


new_price = Real('new_price')
new_name = String('new_name')
new_quantity = Int('new_quantity')

update_product = And(
    ProductExists(pid),
    new_price >= 0,
    new_quantity >= 0,
    Length(new_name) > 0,

    ProductExists_next(pid),
    Products_next(pid) ==
        Product.mk(
            Product.pid(Products(pid)),
            new_name,
            new_price,
            new_quantity,
            Product.sold(Products(pid)),
            Product.seller(Products(pid))
        ),

    frame_products_unchanged(pid),
    frame_all_ratings_unchanged()
)


delete_product = And(
    ProductExists(pid),

    ProductExists_next(pid),
    Products_next(pid) ==
        Product.mk(
            Product.pid(Products(pid)),
            Product.name(Products(pid)),
            Product.price(Products(pid)),
            0,
            0,
            Product.seller(Products(pid))
        ),

    frame_products_unchanged(pid),
    frame_all_ratings_unchanged()
)


rid_new = Const('rid_new', RatingID)
pid_target = Const('pid_target', ProductID)
score_new = Int('score_new')
comment_new = String('comment_new')

add_rating = And(
    ProductExists(pid_target),
    score_new >= 1,
    score_new <= 5,
    Length(comment_new) > 0,

    RatingExists_next(rid_new),
    Ratings_next(rid_new) ==
        Rating.mk(rid_new, pid_target, score_new, comment_new),

    ForAll(
        pid_f,
        And(
            Products_next(pid_f) == Products(pid_f),
            ProductExists_next(pid_f) == ProductExists(pid_f)
        )
    ),

    ForAll(
        rid_f,
        Implies(
            rid_f != rid_new,
            And(
                Ratings_next(rid_f) == Ratings(rid_f),
                RatingExists_next(rid_f) == RatingExists(rid_f)
            )
        )
    )
)

def test_operation(name, transition):
    s = Solver()
    s.add(global_invariants())
    s.add(transition)
    s.add(Not(
        And(
            ForAll(pid, Implies(ProductExists_next(pid),
                                product_invariants(Products_next(pid)))),
            ForAll(rid, Implies(RatingExists_next(rid),
                                rating_invariants(Ratings_next(rid))))
        )
    ))

    print(f"\n▶ Test: {name}")
    if s.check() == sat:
        print("Violates invariants")
        print(s.model())
    else:
        print("Preserves all invariants")


test_operation("Purchase", purchase)
test_operation("UpdateProduct", update_product)
test_operation("DeleteProduct", delete_product)
test_operation("AddRating", add_rating)

print("\n✔ Complete HLF formal verification finished")


▶ Test: Purchase
Preserves all invariants

▶ Test: UpdateProduct
Preserves all invariants

▶ Test: DeleteProduct
Preserves all invariants

▶ Test: AddRating
Preserves all invariants

✔ Complete HLF formal verification finished
